# AI Incidents Pipeline Runner

Loads the configuration from `config/params.yaml` and runs the full pipeline:
ingestion -> preprocessing -> NLP -> sentiment -> analysis -> visualization/report.

In [1]:
from pathlib import Path

import yaml

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.visualization import run_visualization

In [2]:
with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

configure_logging()
set_global_seeds(config["random_state"])

In [3]:
df = load_and_validate(config)
df.head()

2026-08-06 23:33:33,546 [INFO] src.ingestion: Loaded 1605 rows from data\raw\aiid_incidents.csv (encoding=utf-8)
2026-08-06 23:33:33,550 [INFO] src.ingestion: Column 'geo_zone' has 87.5% null values
2026-08-06 23:33:33,552 [INFO] src.ingestion: Column 'tags_list' has 6.8% null values
2026-08-06 23:33:33,573 [INFO] src.ingestion: No duplicate rows found


,Incident ID,event_date,year,title,text_data,deployer,developer,harmed,Editors,Implicated Systems,...,Detrimental Content,Impact on Critical Services,Multiple AI Interaction,Embedded,Entities,Physical Objects,Deployed,Estimated Date,Estimated Harm Quantities,Special Interest Intangible Harm
0,1,2015-05-19 00:00:00,2015,Google’s YouTube Kids App Presents Inappropria...,YouTube’s content filtering and recommendation...,Youtube,Youtube,Minors,619B47Ea5Eed5334Edfa3Bbc,NaN,...,yes,no,no,no,"[{""attributes"": [{""short_name"": ""Entity"", ""val...",no,yes,True,False,yes
1,2,2018-12-05 00:00:00,2018,Warehouse robot ruptures can of bear spray and...,Twenty-four Amazon workers in New Jersey were ...,Amazon,Amazon,Warehouse Workers,619B47Ea5Eed5334Edfa3Bbc,NaN,...,no,no,no,yes,"[{""attributes"": [{""short_name"": ""Entity"", ""val...",yes,yes,False,False,no
2,3,2018-10-27 00:00:00,2018,Crashes with Maneuvering Characteristics Augme...,"A Boeing 737 crashed into the sea, killing 189...",Boeing,Boeing,"Airplane Passengers, Airplane Crew",619B47Ea5Eed5334Edfa3Bbc,NaN,...,no,no,no,yes,"[{""attributes"": [{""short_name"": ""Entity"", ""val...",yes,yes,False,False,no
3,4,2018-03-18 00:00:00,2018,Uber AV Killed Pedestrian in Arizona,An Uber autonomous vehicle (AV) in autonomous ...,Uber,Uber,"Elaine Herzberg, Pedestrians","619B47Ea5Eed5334Edfa3Bbc, 62970Eca16E5E43939C2...",NaN,...,no,no,no,yes,"[{""attributes"": [{""short_name"": ""Entity"", ""val...",yes,no,False,False,no
4,5,2015-07-13 00:00:00,2015,Collection of Robotic Surgery Malfunctions,Study on database reports of robotic surgery m...,"Hospitals, Doctors",Intuitive Surgical,Patients,619B47Ea5Eed5334Edfa3Bbc,NaN,...,no,no,no,yes,"[{""attributes"": [{""short_name"": ""Entity"", ""val...",yes,yes,True,False,no


In [4]:
df = preprocess(df, config)
df.head()

2026-08-06 23:33:33,664 [INFO] src.preprocessing: Dropping 939 rows outside year range [2014, 2023]
2026-08-06 23:33:33,670 [INFO] src.preprocessing: Dropping 544 rows outside regions ['North America', 'Europe', 'Asia']


,Incident ID,event_date,year,title,text_data,deployer,developer,harmed,Editors,Implicated Systems,...,mlb_industries__law enforcement,mlb_industries__manufacturing,mlb_industries__other,mlb_industries__other service activities,"mlb_industries__professional, scientific and technical activities",mlb_industries__public administration,mlb_industries__real estate activities,mlb_industries__transportation and storage,mlb_industries__unclear,mlb_industries__wholesale and retail trade
0,1,2015-05-19,2015,Google’s YouTube Kids App Presents Inappropria...,YouTube’s content filtering and recommendation...,Youtube,Youtube,Minors,619B47Ea5Eed5334Edfa3Bbc,NaN,...,0,0,0,0,0,0,0,0,0,0
1,2,2018-12-05,2018,Warehouse robot ruptures can of bear spray and...,Twenty-four Amazon workers in New Jersey were ...,Amazon,Amazon,Warehouse Workers,619B47Ea5Eed5334Edfa3Bbc,NaN,...,0,0,0,0,0,0,0,0,0,1
2,3,2018-10-27,2018,Crashes with Maneuvering Characteristics Augme...,"A Boeing 737 crashed into the sea, killing 189...",Boeing,Boeing,"Airplane Passengers, Airplane Crew",619B47Ea5Eed5334Edfa3Bbc,NaN,...,0,0,0,0,0,0,0,1,0,0
3,4,2018-03-18,2018,Uber AV Killed Pedestrian in Arizona,An Uber autonomous vehicle (AV) in autonomous ...,Uber,Uber,"Elaine Herzberg, Pedestrians","619B47Ea5Eed5334Edfa3Bbc, 62970Eca16E5E43939C2...",NaN,...,0,0,0,0,0,0,0,1,0,0
4,5,2015-07-13,2015,Collection of Robotic Surgery Malfunctions,Study on database reports of robotic surgery m...,"Hospitals, Doctors",Intuitive Surgical,Patients,619B47Ea5Eed5334Edfa3Bbc,NaN,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

2026-08-06 23:33:33,724 [INFO] src.nlp: Downloading NLTK resource 'wordnet'
2026-08-06 23:33:34,023 [INFO] src.nlp: Downloading NLTK resource 'omw-1.4'


,tokens,mental_health_flag
0,"[youtubes, content, filtering, recommendation,...",0
1,"[twentyfour, amazon, worker, jersey, hospitali...",0
2,"[boeing, 737, crashed, sea, killing, 189, peop...",0
3,"[uber, autonomous, vehicle, autonomous, mode, ...",0
4,"[study, database, robotic, surgery, malfunctio...",0


In [6]:
df = run_sentiment(df, config)
df[["sentiment_score", "sentiment_label"]].head()

2026-08-06 23:33:42,693 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:44,448 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:45,580 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:46,517 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:47,193 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:48,161 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:49,087 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:49,762 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-06 23:33:50,780 [INFO] httpx: HTTP Request: POST

,sentiment_score,sentiment_label
0,0.8,Alta
1,0.7,Alta
2,1.0,Alta
3,0.9,Alta
4,0.7,Alta


In [7]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)

In [8]:
metrics = run_analysis(df, config)

2026-08-06 23:35:22,238 [INFO] src.analysis: Metrics written to outputs\reports\metrics.json


In [9]:
report_path = run_visualization(df, metrics, config)
report_path

2026-08-06 23:35:22,855 [INFO] src.visualization: Saved figure 'regional_evolution' to outputs\figures\regional_evolution.png
2026-08-06 23:35:23,122 [INFO] src.visualization: Saved figure 'cumulative_concentration' to outputs\figures\cumulative_concentration.png
2026-08-06 23:35:23,559 [INFO] src.visualization: Saved figure 'principles_distribution' to outputs\figures\principles_distribution.png
2026-08-06 23:35:24,108 [INFO] src.visualization: Saved figure 'harm_types_distribution' to outputs\figures\harm_types_distribution.png
2026-08-06 23:35:24,515 [INFO] src.visualization: Saved figure 'stakeholders_distribution' to outputs\figures\stakeholders_distribution.png
2026-08-06 23:35:24,772 [INFO] src.visualization: Saved figure 'vulnerable_groups' to outputs\figures\vulnerable_groups.png
2026-08-06 23:35:25,208 [INFO] src.visualization: Saved figure 'harm_chronology' to outputs\figures\harm_chronology.png
2026-08-06 23:35:25,794 [INFO] src.visualization: Saved figure 'top_industries' 

WindowsPath('outputs/reports/executive_report.pdf')

In [10]:
print("Pipeline completado.")

Pipeline completado.
